In [4]:
# importing the required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import kmapper as km
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
# from umap import UMAP
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import os
import joblib

In [5]:
# importing preprocessed data
combined_df = pd.read_csv("../data/combined_cleaned_encoded.csv")

In [6]:
# Scaling 
# Select numerical columns for scaling
num_cols = ['age', 'tenure', 'monthlycharges']

# Initialize the scaler
scaler = StandardScaler() # mean=0, std=1

# Fit the scaler on numeric columns and transform them
df_scaled = combined_df.copy()
df_scaled[num_cols] = scaler.fit_transform(combined_df[num_cols])

print(df_scaled.head())

      customer_id  gender       age    tenure  monthlycharges  paymentmethod  \
0  TEL_7590-VHVEG       0 -0.371853 -1.511772       -1.155154              5   
1  TEL_5575-GNVDE       1 -0.103853  0.076123       -0.136535              6   
2  TEL_3668-QPYBK       1 -1.376852 -1.463654       -0.253056              6   
3  TEL_7795-CFOCW       1 -1.443852  0.605421       -0.687190              0   
4  TEL_9237-HQITU       0 -1.577852 -1.463654        0.380292              5   

   churn  industry  
0    0.0         2  
1    0.0         2  
2    1.0         2  
3    0.0         2  
4    1.0         2  


In [7]:
#df_scaled copy

df_scaled_copy = df_scaled.copy()

In [8]:
#dropping the customer_id column since it is not a numerical column
df_scaled = df_scaled.drop(columns=["customer_id", "churn"])

In [9]:
#preparing scaled data (converting from pandas to numpy to get numerical matrix form)
df_data = df_scaled.values

# TDA

In [10]:
#creating mapper object (verbose=1 shows progess messages)
mapper = km.KeplerMapper(verbose=1)

KeplerMapper(verbose=1)


In [11]:
#choosing the projection
#we take the scaled dataset, reduce it to 2D using TSNE and it gives projected points so Mapper can 
#build a topological graph
projected = mapper.fit_transform(df_data, projection=TSNE(n_components=2))

..Composing projection pipeline of length 1:
	Projections: TSNE()
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (21129, 6)

..Projecting data using: 
	TSNE(verbose=1)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 21129 samples in 0.014s...
[t-SNE] Computed neighbors for 21129 samples in 0.530s...
[t-SNE] Computed conditional probabilities for sample 1000 / 21129
[t-SNE] Computed conditional probabilities for sample 2000 / 21129
[t-SNE] Computed conditional probabilities for sample 3000 / 21129
[t-SNE] Computed conditional probabilities for sample 4000 / 21129
[t-SNE] Computed conditional probabilities for sample 5000 / 21129
[t-SNE] Computed conditional probabilities for sample 6000 / 21129
[t-SNE] Computed conditional probabilities for sample 7000 / 21129
[t-SNE] Computed conditional probabilities for sample 8000 / 21129
[t-SNE] Computed conditional probabilities for sample 9000 / 21129
[t-SNE] Computed conditional probabilities for sample

In [12]:
#creating the topological network
graph=mapper.map(projected, df_data, clusterer=KMeans(n_clusters=4), cover=km.Cover(n_cubes=10, perc_overlap=0.3))

Mapping on data shaped (21129, 6) using lens shaped (21129, 2)

Creating 100 hypercubes.

Created 963 edges and 340 nodes in 0:00:00.306032.


In [13]:
#extracting TDA features

#graph['nodes'] includes list of sample indices in each node
nodes = graph['nodes']

#creating a feature matrix where each sample gets a binary vector for node members
tda_features = np.zeros((df_data.shape[0], len(nodes)))

for i, node_samples in enumerate(nodes.values()):
    tda_features[node_samples, i] = 1 #marking 1 if the sample belongs to this node


#turning the above into one-hot encoding
tda_features_df = pd.DataFrame(tda_features, columns=[f"tda_node_{i}" for i in range(len(nodes))])

#combining with original data
df_final = pd.concat([pd.DataFrame(df_data).reset_index(drop=True), tda_features_df.reset_index(drop=True)], axis=1)

In [25]:
X_train.head()

,0,1,2,3,4,5,tda_node_0,tda_node_1,tda_node_2,tda_node_3,...,tda_node_330,tda_node_331,tda_node_332,tda_node_333,tda_node_334,tda_node_335,tda_node_336,tda_node_337,tda_node_338,tda_node_339
16197,0.0,1.370145,1.327191,-1.265285,8.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1103,1.0,1.236145,1.038483,1.677058,5.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14115,1.0,-0.237853,0.172359,-0.942785,7.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
413,1.0,0.097147,0.894129,1.006123,5.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12248,0.0,-0.974852,-1.174946,-0.747330,2.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# nodes: dictionary of node_id -> list of sample indices
# df_data or X_scaled: your numeric data as a NumPy array

tda_node_centers = []

for node_id, sample_indices in nodes.items():
    # If df_data is a NumPy array, just index with sample_indices
    center = df_data[sample_indices].mean(axis=0)  # NO .iloc here
    tda_node_centers.append(center)

tda_node_centers = np.vstack(tda_node_centers)
print(tda_node_centers)

# Save to a file
np.save("../models/tda_node_centers.npy", tda_node_centers)
print("Saved TDA node centers:", tda_node_centers.shape)


[[ 1.          0.25347972  1.64797751 -0.28124617  0.          2.        ]
 [ 1.          0.25347972  1.85648888  0.26721689  0.          2.        ]
 [ 1.          0.05247995  1.58382016  0.21678711  0.          2.        ]
 ...
 [ 0.          0.88932216 -0.94001664 -0.08455343  9.          0.        ]
 [ 0.          1.30708641 -0.30599109 -0.90745254  9.          0.        ]
 [ 0.          1.02770109 -1.25246921 -0.89281433  9.          0.        ]]
Saved TDA node centers: (340, 6)


In [26]:
# --- save final feature column list for sanity (optional but useful) ---
# X_train = pd.concat([encoded_cats_df, scaled_numeric_df, tda_features_df], axis=1)  
# should be the X you trained 

save_path = "../backend/artifacts"
feature_cols = list(X_train.columns)
joblib.dump(feature_cols, os.path.join(save_path, "feature_columns.pkl"))
print("Saved feature column list, length:", len(feature_cols))

Saved feature column list, length: 346


#load this in your backend:


tda_nodes = np.load("../models/tda_node_centers.npy", allow_pickle=True)


In [18]:
df_scaled_copy = df_scaled_copy["churn"]

In [19]:
df_scaled_copy.head()

0    0.0
1    0.0
2    1.0
3    0.0
4    1.0
Name: churn, dtype: float64

In [20]:
df_final.head()

,0,1,2,3,4,5,tda_node_0,tda_node_1,tda_node_2,tda_node_3,...,tda_node_330,tda_node_331,tda_node_332,tda_node_333,tda_node_334,tda_node_335,tda_node_336,tda_node_337,tda_node_338,tda_node_339
0,0.0,-0.371853,-1.511772,-1.155154,5.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,-0.103853,0.076123,-0.136535,6.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,-1.376852,-1.463654,-0.253056,6.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,-1.443852,0.605421,-0.687190,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,-1.577852,-1.463654,0.380292,5.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
#spliting into features (X) and target (Y)
Y = df_scaled_copy #column named 5 is churn column
X = df_final

In [22]:
#Y.rename(columns={"churn": "6"}, inplace=True)
Y.name = "6"

In [23]:
#train-test split

X_train, X_test, Y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

In [27]:
print("Training X shape:", X_train.shape)
print("Columns:", list(X_train.columns))

Training X shape: (16903, 346)
Columns: ['0', '1', '2', '3', '4', '5', 'tda_node_0', 'tda_node_1', 'tda_node_2', 'tda_node_3', 'tda_node_4', 'tda_node_5', 'tda_node_6', 'tda_node_7', 'tda_node_8', 'tda_node_9', 'tda_node_10', 'tda_node_11', 'tda_node_12', 'tda_node_13', 'tda_node_14', 'tda_node_15', 'tda_node_16', 'tda_node_17', 'tda_node_18', 'tda_node_19', 'tda_node_20', 'tda_node_21', 'tda_node_22', 'tda_node_23', 'tda_node_24', 'tda_node_25', 'tda_node_26', 'tda_node_27', 'tda_node_28', 'tda_node_29', 'tda_node_30', 'tda_node_31', 'tda_node_32', 'tda_node_33', 'tda_node_34', 'tda_node_35', 'tda_node_36', 'tda_node_37', 'tda_node_38', 'tda_node_39', 'tda_node_40', 'tda_node_41', 'tda_node_42', 'tda_node_43', 'tda_node_44', 'tda_node_45', 'tda_node_46', 'tda_node_47', 'tda_node_48', 'tda_node_49', 'tda_node_50', 'tda_node_51', 'tda_node_52', 'tda_node_53', 'tda_node_54', 'tda_node_55', 'tda_node_56', 'tda_node_57', 'tda_node_58', 'tda_node_59', 'tda_node_60', 'tda_node_61', 'tda_node

In [24]:
X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)


In [ ]:
#Logistic Regression

log_reg = LogisticRegression(max_iter=1000, random_state=42) #creating the model
log_reg.fit(X_train, Y_train) #training the model (.fit() means learn from the dataset)
y_pred_lr = log_reg.predict(X_test) #make predictions
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr)) #evaluate the model's performance
print(classification_report(y_test, y_pred_lr)) #metrics report

In [ ]:
#Random Forest

rf = RandomForestClassifier(n_estimators=200, random_state=42) #creating the model
rf.fit(X_train, Y_train) #training the model (.fit() means learn from the dataset)
y_pred_rf = rf.predict(X_test) #make predictions
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf)) #evaluate the model's performance
print(classification_report(y_test, y_pred_rf)) #metrics report

In [ ]:
# XGBoost

xgb = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=42) #creating the model
xgb.fit(X_train, Y_train) #training the model (.fit() means learn from the dataset)
y_pred_xgb = xgb.predict(X_test) #make predictions
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb)) #evaluate the model's performance
print(classification_report(y_test, y_pred_xgb)) #metrics report

------------------------------------------------------------------------------------------

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Assume these are your trained models
# logistic_regression, random_forest, xgboost_model
# Make sure they are fitted already

# Create a voting ensemble (soft voting uses predicted probabilities)
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', log_reg),
        ('rf', rf),
        ('xgb', xgb)
    ],
    voting='soft'  # use 'hard' for majority vote
)

# Fit ensemble on training data
ensemble_model.fit(X_train, Y_train)

# Predict on test data
y_pred_ensemble = ensemble_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred_ensemble)
cm = confusion_matrix(y_test, y_pred_ensemble)
report = classification_report(y_test, y_pred_ensemble)

print(f"Ensemble Accuracy: {accuracy:.4f}")
print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(report)


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -----------------------------------------
# Base models (these should already be trained)
# -----------------------------------------
# rf  = RandomForestClassifier(...).fit(X_train, y_train)
# lr  = LogisticRegression(...).fit(X_train, y_train)
# xgb = XGBClassifier(...).fit(X_train, y_train)

# IMPORTANT: If not trained already, remove .fit() above and let stacking train them.

# -----------------------------------------
# Stacking Ensemble
# -----------------------------------------
estimators = [
    ('lr', log_reg),
    ('rf', rf),
    ('xgb', xgb)
]

# Meta-model (best choice for classification stacking)
meta_model = LogisticRegression(max_iter=200, class_weight='balanced')

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    stack_method='predict_proba',
    passthrough=True   # gives meta-model BOTH original features + base model outputs
)

# Train stacking model
stack_model.fit(X_train, Y_train)

# Predict
y_pred_stack = stack_model.predict(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred_stack)
cm  = confusion_matrix(y_test, y_pred_stack)
rep = classification_report(y_test, y_pred_stack)

print("Stacking Ensemble Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", rep)


-----------------------------------------------------------------------------------------

## SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# BEFORE SMOTE
print("Before SMOTE:", Counter(Y_train))

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, Y_train)

# AFTER SMOTE
print("After SMOTE:", Counter(y_train_smote))


In [ ]:
#Logistic Regression

log_reg_smote = LogisticRegression(max_iter=1000, random_state=42) #creating the model
log_reg_smote.fit(X_train_smote, y_train_smote) #training the model (.fit() means learn from the dataset)
y_pred_lr = log_reg_smote.predict(X_test) #make predictions
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr)) #evaluate the model's performance
print(classification_report(y_test, y_pred_lr)) #metrics report

In [ ]:
#Random Forest

rf_smote = RandomForestClassifier(n_estimators=200, random_state=42) #creating the model
rf_smote.fit(X_train_smote, y_train_smote) #training the model (.fit() means learn from the dataset)
y_pred_rf_smote = rf_smote.predict(X_test) #make predictions
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf_smote)) #evaluate the model's performance
print(classification_report(y_test, y_pred_rf_smote)) #metrics report

In [ ]:
# XGBoost

xgb_smote = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=42) #creating the model
xgb_smote.fit(X_train_smote, y_train_smote) #training the model (.fit() means learn from the dataset)
y_pred_xgb_smote = xgb_smote.predict(X_test) #make predictions
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb_smote)) #evaluate the model's performance
print(classification_report(y_test, y_pred_xgb_smote)) #metrics report

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -----------------------------------------
# Base models (these should already be trained)
# -----------------------------------------
# rf  = RandomForestClassifier(...).fit(X_train, y_train)
# lr  = LogisticRegression(...).fit(X_train, y_train)
# xgb = XGBClassifier(...).fit(X_train, y_train)

# IMPORTANT: If not trained already, remove .fit() above and let stacking train them.

# -----------------------------------------
# Stacking Ensemble
# -----------------------------------------
estimators = [
    ('lr', log_reg_smote),
    ('rf', rf_smote),
    ('xgb', xgb_smote)
]

# Meta-model (best choice for classification stacking)
meta_model_smote = LogisticRegression(max_iter=200, class_weight='balanced')

stack_model_smote = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model_smote,
    stack_method='predict_proba',
    passthrough=True   # gives meta-model BOTH original features + base model outputs
)

# Train stacking model
stack_model_smote.fit(X_train_smote, y_train_smote)

# Predict
y_pred_stack_smote = stack_model_smote.predict(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred_stack_smote)
cm  = confusion_matrix(y_test, y_pred_stack_smote)
rep = classification_report(y_test, y_pred_stack_smote)

print("Stacking Ensemble Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", rep)


## using more models

In [38]:
#tuned XGBoost

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_smote, y_train_smote)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


XGBoost (Extreme Gradient Boosting)
this model builds a series of decision trees where each tree tries to correct the mistakes of the previous one. It is very fast and handles missing values and works well on structured tabular data.
We used this model since it corrects mistakes iteratively. It also gives high accuracy on imbalanced datasets

In [40]:
#LightGBM

from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(X_train_smote, y_train_smote)


[LightGBM] [Info] Number of positive: 10054, number of negative: 10054
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005655 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1483
[LightGBM] [Info] Number of data points in the train set: 20108, number of used features: 337
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


LightGBM (Light Gradient Boosting Machine)
Similar to XGBoost but even faster and more efficient with large datasets.
It uses leaf-wise tree growth, which captures complex patterns better
we used this because its faster than XGBoost and handles large datasets efficiently. It also captures patterns well

In [41]:
#CatBoost

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    eval_metric='Accuracy',
    verbose=0,
    random_seed=42
)

cat_model.fit(X_train_smote, y_train_smote)


CatBoost (categorical boosting)
Designed to handle categorical features like customer type, region etc. directly without heavy preprocessing.
it gives strong results on imbalanced dataset
We used this because it handles categorical variables naturally, it is strong on imbalanced datasets and often gives best results out-of-the-box

In [43]:
#Evaluating each model

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

models = {
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "CatBoost": cat_model
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print("\n======", name, "======")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))



====== XGBoost ======
Accuracy: 0.6796024609559868
[[1603  911]
 [ 443 1269]]
              precision    recall  f1-score   support

         0.0       0.78      0.64      0.70      2514
         1.0       0.58      0.74      0.65      1712

    accuracy                           0.68      4226
   macro avg       0.68      0.69      0.68      4226
weighted avg       0.70      0.68      0.68      4226


====== LightGBM ======
Accuracy: 0.6817321344060577
[[1733  781]
 [ 564 1148]]
              precision    recall  f1-score   support

         0.0       0.75      0.69      0.72      2514
         1.0       0.60      0.67      0.63      1712

    accuracy                           0.68      4226
   macro avg       0.67      0.68      0.68      4226
weighted avg       0.69      0.68      0.68      4226


====== CatBoost ======
Accuracy: 0.6819687647893989
[[1598  916]
 [ 428 1284]]
              precision    recall  f1-score   support

         0.0       0.79      0.64      0.70      251

In [45]:
#Ensemble model

from sklearn.ensemble import VotingClassifier

super_ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('cat', cat_model)
    ],
    voting='soft',      # use probability averaging
    weights=[2, 2, 3]   # CatBoost is usually strongest
)

super_ensemble.fit(X_train_smote, y_train_smote)

y_pred_ens = super_ensemble.predict(X_test)

print("\n===== SUPER ENSEMBLE RESULTS =====")
print("Accuracy:", accuracy_score(y_test, y_pred_ens))
print(confusion_matrix(y_test, y_pred_ens))
print(classification_report(y_test, y_pred_ens))


[LightGBM] [Info] Number of positive: 10054, number of negative: 10054
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002514 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1483
[LightGBM] [Info] Number of data points in the train set: 20108, number of used features: 337
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000

===== SUPER ENSEMBLE RESULTS =====
Accuracy: 0.6822053951727401
[[1643  871]
 [ 472 1240]]
              precision    recall  f1-score   support

         0.0       0.78      0.65      0.71      2514
         1.0       0.59      0.72      0.65      1712

    accuracy                           0.68      4226
   macro avg       0.68      0.69      0.68      4226
weighted avg       0.70      0.68      0.69      4226



Ensemble learning helps us because
- Customer churn is imbalanced
- Individual models may miss churners (they may end up giving false negatives)
- ensemble increases chances of detecting churners correctly
- it can be helpful in catching potential churners which inturn saves revenue of businesses

Accuracy -> tells the overall correctness
recall for churn -> shows percentage of actual churners
f1-score -> balance between precision and recall. Gives a hoslitic measure of churn prediction quality

Why does our approach work??
- it helps handle numerical and categorical features
- works with imbalanced data (with the help of SMOTE)
- ensemble ensures robust, repeatable results are not dependent on a single model

## saving models and datasets

In [64]:
import os
import joblib

save_path = "../notebooks/final_models"
os.makedirs(save_path, exist_ok=True)

#df_final.to_csv(os.path.join(save_path, "combined_cleaned_TDA.csv"), index=False)

# Save each model
joblib.dump(log_reg_smote, 'final_models/log_reg_smote.pkl')
joblib.dump(rf_smote, 'final_models/srf_smote.pkl')
joblib.dump(xgb_smote, 'final_models/xgb_smote.pkl')
joblib.dump(lgb_model, 'final_models/lgb_model.pkl')
joblib.dump(cat_model, 'final_models/cat_model.pkl')
joblib.dump(super_ensemble, 'final_models/super_ensemble.pkl')
joblib.dump(stack_model_smote, 'final_models/stack_model_smote.pkl')

print("All models saved successfully!")


All models saved successfully!


In [66]:
#saving scalar
joblib.dump(scaler, 'final_models/scaler.pkl')


['final_models/scaler.pkl']

In [65]:
#saving dataset

save_path = "../data/train_test_data"
os.makedirs(save_path, exist_ok=True)

# Save train and test sets (or the full dataset)
X_train.to_csv(os.path.join(save_path, 'X_train.csv'), index=False)
Y_train.to_csv(os.path.join(save_path, 'y_train.csv'), index=False)
X_test.to_csv(os.path.join(save_path, 'X_test.csv'), index=False)
y_test.to_csv(os.path.join(save_path, 'y_test.csv'), index=False)

# Optionally save the SMOTE-balanced train set
X_train_smote.to_csv(os.path.join(save_path, 'X_train_smoteote.csv'), index=False)
y_train_smote.to_csv(os.path.join(save_path, 'y_train_smoteote.csv'), index=False)

print("All datasets saved successfully!")


All datasets saved successfully!


In [ ]:
try:
    X_train_df.to_csv(os.path.join(save_path, "X_train_bal.csv"), index=False)
    Y_train_df.to_csv(os.path.join(save_path, "Y_train_bal.csv"), index=False)
    X_test_df.to_csv(os.path.join(save_path, "X_test.csv"), index=False)
    y_test_df.to_csv(os.path.join(save_path, "y_test.csv"), index=False)
    print("SMOTE based train/test sets saved")
except Exception as e:
    print("Could not save", e)

In [67]:
super_ensemble.predict_proba(X_test)[:5]

array([[0.3813644 , 0.6186356 ],
       [0.42725522, 0.57274478],
       [0.77815368, 0.22184631],
       [0.66754585, 0.33245414],
       [0.60207169, 0.39792832]])